This notebook is for testing purposes to check capability of using Machine Learning for Marketing insights.

# Library import

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px


# Importing Data and cleaning


In [2]:
main_data = pd.read_csv("ML_data.csv",sep=",")
main_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36782 entries, 0 to 36781
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   next_purchase_after_30d  36782 non-null  int64  
 1   Transaction_Value        36782 non-null  float64
 2   Coupon_Status            36782 non-null  object 
 3   Gender                   36782 non-null  object 
 4   Tenure_Months            36782 non-null  int64  
 5   Delivery_Charges         36782 non-null  float64
dtypes: float64(2), int64(2), object(2)
memory usage: 1.7+ MB


Double checking for null values as data was already aggregated in bigQuery using SQL


In [3]:
main_data.isnull().sum()

next_purchase_after_30d    0
Transaction_Value          0
Coupon_Status              0
Gender                     0
Tenure_Months              0
Delivery_Charges           0
dtype: int64

As we have `Coupon_Status` and `Gender` as objects and not numeric columns, we can quickly change into binary both.

In [4]:
coupon_used_dict = {"Used":1,"Unused":0}
gender_dict = {"F":1,"M":0}
main_data["Coupon_Status_binary"] = main_data["Coupon_Status"].map(coupon_used_dict)
main_data["Gender_binary"] = main_data["Gender"].map(gender_dict)
main_data.head()

,next_purchase_after_30d,Transaction_Value,Coupon_Status,Gender,Tenure_Months,Delivery_Charges,Coupon_Status_binary,Gender_binary
0,1,45.54,Used,F,27,26.43,1,1
1,0,3.95,Used,F,27,19.99,1,1
2,0,17.59,Used,F,27,6.00,1,1
3,0,3.99,Used,F,27,6.00,1,1
4,0,41.81,Used,F,27,6.50,1,1


# EDA

A quick EDA to check what can be seen from few charts


In [5]:
value_coupon = px.box(main_data,x="Coupon_Status", y="Transaction_Value"
                      ,color="next_purchase_after_30d", title="Value on Coupon usage",
                       labels={"Coupon_Status":"Coupon Status", "Transaction_Value":"Transaction Value","next_purchase_after_30d":"Bought again after 30 days"})
value_coupon.show()

We see that there quite a bit outliers with transaction values. while transactions that have done later on has lower count and generaly smaller outlier count.

In [6]:
value_gender = px.box(main_data,x="Gender", y="Transaction_Value"
                      ,color="next_purchase_after_30d", title="Value by Gender",
                       labels={"Transaction_Value":"Transaction Value","next_purchase_after_30d":"Bought again after 30 days"})
value_gender.show()

Similar story when we divide by gender. 

In [7]:
delivery_costs_Coupon =  px.box(main_data,x="Coupon_Status", y="Delivery_Charges"
                      ,color="next_purchase_after_30d", title="Delivery charges on Coupon usage",
                       labels={"Coupon_Status":"Coupon Status", "Delivery_Charges":"Delivery Charge","next_purchase_after_30d":"Bought again after 30 days"})
delivery_costs_Coupon.show()

Repetitive trend from delivery charges. It makes sense as delivery charge could vary by transaction value.

It was a short EDA as goal of this notebook focuses more on seeing if ML could be applied later on wif we collect more data.

# Modeling

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score, auc



Let's drop columns that we already have encoded.

In [9]:
ml_df = main_data.drop(columns=["Coupon_Status","Gender"])
ml_df.head()

,next_purchase_after_30d,Transaction_Value,Tenure_Months,Delivery_Charges,Coupon_Status_binary,Gender_binary
0,1,45.54,27,26.43,1,1
1,0,3.95,27,19.99,1,1
2,0,17.59,27,6.00,1,1
3,0,3.99,27,6.00,1,1
4,0,41.81,27,6.50,1,1


In [10]:
y=ml_df["next_purchase_after_30d"]
X=ml_df.drop(columns=["next_purchase_after_30d"])
y.shape

(36782,)

Spliting dataset into train and test ones. Split will go 70/30 as we have enough data to have a bit more rows for test set.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [12]:
model = LogisticRegression(fit_intercept=True, class_weight="balanced", max_iter=1000)

In [13]:
model.fit(X_train,y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [14]:
y_train_pred = model.predict(X_train)
print(classification_report(y_train,y_train_pred))

              precision    recall  f1-score   support

           0       0.99      0.69      0.81     24064
           1       0.16      0.87      0.28      1683

    accuracy                           0.70     25747
   macro avg       0.58      0.78      0.54     25747
weighted avg       0.93      0.70      0.78     25747



Model is performing good. Even with imbalanced dataset.

In [15]:
train_conf_matrix = confusion_matrix(y_train,y_train_pred)
train_conf_matrix

array([[16602,  7462],
       [  213,  1470]])

We can identify those who will order after 30days again.

In [16]:
y_pred = model.predict(X_test)

In [17]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.68      0.81     10353
           1       0.15      0.87      0.26       682

    accuracy                           0.69     11035
   macro avg       0.57      0.78      0.53     11035
weighted avg       0.94      0.69      0.77     11035



Very close metrics from our testing set. Indicating model performs well with very small amount of features we have.

In [18]:
test_conf_matrix = confusion_matrix(y_test,y_pred)
test_conf_matrix

array([[7074, 3279],
       [  88,  594]])

In [19]:
y_score = model.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_score)

Roc_Curve_fig = px.area(
    x=fpr, y=tpr,
    title=f'ROC Curve (AUC={auc(fpr, tpr):.4f})',
    labels=dict(x='False Positive Rate', y='True Positive Rate'),
    width=700, height=500
)
Roc_Curve_fig.add_shape(
    type='line', line=dict(dash='dash'),
    x0=0, x1=1, y0=0, y1=1
)

Roc_Curve_fig.update_yaxes(scaleanchor="x", scaleratio=1)
Roc_Curve_fig.update_xaxes(constrain='domain')
Roc_Curve_fig.show()

# Conclusion and further investigation

Goal of this ML attempt was to check if it was applicable for future use. There is definitely potential.

Main issue is very small feature amount. Additional features and implementation of our marketing strategies into ML model could help us identify customers faster and put us ahead for determining when we should interact with customers.